In [ ]:
import pandas as pd
from keras.layers import Flatten , Dense  , Embedding , Input , Concatenate , Dropout , BatchNormalization
from keras.layers import  ReLU , LeakyReLU
from keras.optimizers import Adam , SGD , AdamW
from keras.regularizers import l2 
from keras.callbacks import EarlyStopping
from keras.losses import BinaryCrossentropy
from keras.models import Model
import keras_tuner as kt
from tensorflow.random import set_seed

In [ ]:
x_train = pd.read_csv('./datas/Out_Stage3/x_train')
x_valid = pd.read_csv('./datas/Out_Stage3/x_valid')
y_train = pd.read_csv('./datas/Out_Stage3/y_train')
y_valid = pd.read_csv('./datas/Out_Stage3/y_valid')

In [ ]:
numeric_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','AGE','BILL_AMT1','BILL_AMT2',
                'BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

x_train['SEX'] = x_train['SEX']-1
x_valid['SEX'] = x_valid['SEX']-1


def get_layers ():
    input_marriage = Input(shape=(1,) , name='marriage')
    embeding_marriage = Embedding(4,4 // 2 ,name = 'marriage_emb')(input_marriage)
    layer_marriage = Flatten()(embeding_marriage)

    input_education = Input(shape=(1,) , name='education')
    embeding_education = Embedding(7,7 // 2 ,name = 'education_emb')(input_education)
    layer_education = Flatten()(embeding_education)

    input_sex = Input(shape=(1,) , name='sex')
    embeding_sex = Embedding(2,2 // 2 ,name = 'sex_emb')(input_sex)
    layer_sex = Flatten()(embeding_sex)

    numeric_input = Input(shape=(19,))
    layer_numeric = Dense(32, activation='relu')(numeric_input)

    return layer_education , layer_marriage , layer_numeric , layer_sex , input_marriage , input_education , input_sex , numeric_input

In [ ]:
set_seed(42)
class MyHyperModel(kt.HyperModel):
    def build(self,hp):
            n_hidden = hp.Int("n_hidden" , min_value = 1 , max_value = 5 ,default = 2)
            n_neurons = hp.Choice("n_neurons" ,values = [32,64,128])
            n_dropout = hp.Choice("n_dropout" ,values = [0.3,0.4,0.45,0.5,0.55])
            learning_rate = 0.002 
            l2_rate = hp.Float("l2_rate" , min_value=1e-5,max_value = 1e-2 , sampling = "log")
            optimizer = hp.Choice("optimizer" ,values = ["sgd" , "adam" , "adamw"])
            activation = hp.Choice("activation" ,values = ['relu' , 'lrelu'])

            if optimizer == "sgd":
                optimizer = SGD(learning_rate=learning_rate)
            elif optimizer == "adam":
                optimizer = Adam(learning_rate=learning_rate)
            else:
                 optimizer = AdamW(learning_rate=learning_rate ,)

            layer_education , layer_marriage , layer_numeric , layer_sex , input_marriage , input_education , input_sex , numeric_input = get_layers()
            
            concat = Concatenate()([layer_marriage,layer_sex,layer_education ,layer_numeric])


            output = Dense(1,activation='sigmoid')

            layer_main = Dense(n_neurons, kernel_regularizer=l2(l2_rate))(concat)
            layer_main = BatchNormalization()(layer_main)
            layer_main = ReLU()(layer_main) if activation=="relu" else LeakyReLU()(layer_main)
            layer_main = Dropout(n_dropout)(layer_main)
            
            for _ in range(n_hidden-1):
                 layer_main = Dense(n_neurons, kernel_regularizer=l2(l2_rate))(layer_main)
                 layer_main = BatchNormalization()(layer_main)
                 layer_main = ReLU()(layer_main) if activation=="relu" else LeakyReLU()(layer_main)
                 layer_main = Dropout(n_dropout)(layer_main)
            

            layer_output = output(layer_main)

            model_tuned = Model(inputs=[input_sex, input_marriage , input_education , numeric_input] , outputs =[layer_output])

            model_tuned.compile(optimizer=optimizer , loss=BinaryCrossentropy() ,metrics=['accuracy'])


            return model_tuned
    
    def fit(self, hp,model , x, y,**kwargs):
        
        return model.fit(x,y,**kwargs)




In [ ]:
hyberband_tuner = kt.Hyperband(
    MyHyperModel(),
    objective='val_accuracy',
    max_epochs=30,
    factor=3, 
    hyperband_iterations= 3,
    overwrite = True,
    directory = "my_hyperband_save",
    project_name="my_hyperband_save",
    seed=42
)

In [ ]:
hyberband_tuner.search(
    [x_train['SEX'],x_train['MARRIAGE'],x_train['EDUCATION'],x_train[numeric_cols]] ,
    y_train,
    validation_data = (
        [x_valid['SEX'],x_valid['MARRIAGE'],x_valid['EDUCATION'],x_valid[numeric_cols]],
        y_valid
        ),
    callbacks=[
        EarlyStopping(
            patience=5,
            restore_best_weights=True
        )
    ]
)

In [ ]:
model_fine_tuned = hyberband_tuner.get_best_models(1)[0]

In [ ]:
best_params = hyberband_tuner.get_best_hyperparameters(num_trials=1)[0]

In [ ]:
best_params.values